In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format("parquet")\
        .load("abfss://bronze@devnetflixdatalake.dfs.core.windows.net/netflix_titles")

In [0]:
display(df)

In [0]:
df = df.fillna({"duration_minutes":50,"duration_seasons":1})
display(df)

In [0]:
df = df.withColumn("duration_minutes", col("duration_minutes").try_cast(IntegerType()))\
            .withColumn("duration_seasons", col("duration_seasons").try_cast(IntegerType()))


In [0]:
display(df)

In [0]:
df = df.withColumn("shortTitle",split(col("title"), ":")[0])

In [0]:
df = df.withColumn("category",when(col("type") == "TV Show", "small_screen")\
        .when(col("type") == "Movie", "big_screen")\
        .otherwise("other"))

In [0]:
display(df)

In [0]:
df = df.drop(col("_rescued_data"))

In [0]:
df = df.dropna("any")

In [0]:
display(df)

In [0]:
df = df.withColumn("cumulative_sum",sum(col("duration_minutes")).over(Window.orderBy(col("release_year"))))

In [0]:
display(df)

In [0]:
df_agg = df.groupBy("release_year").agg(sum("duration_minutes").alias("sum_minutes")).orderBy(col("release_year"))
df_agg.display()

Databricks visualization. Run in Databricks to view.

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("abfss://silver@devnetflixdatalake.dfs.core.windows.net/netflix_titles")